# Reshaping demographic data using python 

In [324]:
import pandas as pd
import numpy as np

In [325]:
demo = pd.read_csv("C:/Users/cangu/Documents/recess minutes 2025\cenroll2324.csv",
                   encoding='ISO-8859-1',
                   na_values = '*')

<>:1: SyntaxWarning: invalid escape sequence '\c'
<>:1: SyntaxWarning: invalid escape sequence '\c'
C:\Users\cangu\AppData\Local\Temp\ipykernel_20988\3108139213.py:1: SyntaxWarning: invalid escape sequence '\c'
  demo = pd.read_csv("C:/Users/cangu/Documents/recess minutes 2025\cenroll2324.csv",


In [326]:
# Keeping only school level data
demo = demo.loc[demo.AggregateLevel=='S']

# Creating cdscode variable
demo = demo.astype({'DistrictCode':'int64','SchoolCode':'int64'})
demo['cdscode'] = demo['CountyCode'].astype(str) + '' +  demo['DistrictCode'].astype(str) + '' + demo['SchoolCode'].astype(str)
demo = demo.astype({'cdscode':'int64'}) 
demo['ReportingCategory'] = demo['ReportingCategory'].astype('str')

# Keeping only necessary variables
demo = demo.drop(columns=['AcademicYear','AggregateLevel','CountyCode','DistrictCode','SchoolCode','Charter'])

## Race data

In [327]:
# Keeping only race data
race = demo[demo.ReportingCategory.str.startswith('R')]

# Creating new race variable
def label_race(row):
    if row['ReportingCategory'] == 'RB':
        return 1
    if row['ReportingCategory'] == 'RI':
        return 2
    if row['ReportingCategory'] == 'RA':
        return 3
    if row['ReportingCategory'] == 'RF':
        return 4
    if row['ReportingCategory'] == 'RH':
        return 5
    if row['ReportingCategory'] == 'RD':
        return 6
    if row['ReportingCategory'] == 'RP':
        return 7
    if row['ReportingCategory'] == 'RT':
        return 8
    if row['ReportingCategory'] == 'RW':
        return 9
    return
col = race.apply(label_race, axis = 1)
race = race.assign(race=col.values)
race = race.drop(columns='ReportingCategory')

# Reshaping from long to wide 
race = pd.pivot(race, index=['CountyName', 'DistrictName','SchoolName','cdscode'], columns='race', values='CumulativeEnrollment')
race.reset_index(level=None,drop=False,inplace=True,col_level=0,col_fill='')
race.columns = ['county','district','school','cdscode','afriameri',
                'native','asian','filipino','hispanic','notreport',
                'pacisland','multiracial','white']

## Gender data

In [328]:
# Keeping only gender data
gender = demo[demo.ReportingCategory.str.startswith('G')]

# Creating new gender variable
def label_gender(row):
    if row['ReportingCategory'] == 'GM':
        return 1
    if row['ReportingCategory'] == 'GF':
        return 2
    if row['ReportingCategory'] == 'GX':
        return 3
    if row['ReportingCategory'] == 'GZ':
        return 4
    return
col = gender.apply(label_gender, axis = 1)
gender = gender.assign(gender=col.values)
gender = gender.drop(columns='ReportingCategory')

# Reshaping from long to wide 
gender = pd.pivot(gender, index=['CountyName', 'DistrictName','SchoolName','cdscode'], columns='gender', values='CumulativeEnrollment')
gender.reset_index(level=None,drop=False,inplace=True,col_level=0,col_fill='')
gender.columns = ['county','district','school','cdscode','male','female','nonbinary','miss_gen']

## Other student data

In [329]:
# Keeping only other student data
other = demo[demo.ReportingCategory.str.startswith('S')]

# Creating new other variable
def label_other(row):
    if row['ReportingCategory'] == 'SE':
        return 1
    if row['ReportingCategory'] == 'SD':
        return 2
    if row['ReportingCategory'] == 'SS':
        return 3
    if row['ReportingCategory'] == 'SM':
        return 4
    if row['ReportingCategory'] == 'SF':
        return 5
    if row['ReportingCategory'] == 'SH':
        return 6
    return
col = other.apply(label_other, axis = 1)
other = other.assign(other=col.values)
other = other.drop(columns='ReportingCategory')

# Reshaping from long to wide 
other = pd.pivot(other, index=['CountyName', 'DistrictName','SchoolName','cdscode'], columns='other', values='CumulativeEnrollment')
other.reset_index(level=None,drop=False,inplace=True,col_level=0,col_fill='')
other.columns = ['county','district','school','cdscode','ell','disability','socioeco','migrant','foster','homeless']

## Merge all data frames

In [331]:
merge1 = pd.merge(race, gender, how='outer',on=['county','district','school','cdscode'])
demo_clean = pd.merge(merge1, other, how='outer',on=['county','district','school','cdscode'])

# Export to processed data folder
demo_clean.to_csv("C:/Users/cangu/Documents/recess minutes 2025/SB291-Recess-Data/processed data/demo_2324_clean.csv", index=False)